# Unified Indian Banking Credit Risk Pipeline

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from rapidfuzz import process, utils
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# File Paths
file_3_5 = 'Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'
file_restructuring = '13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx'
file_npa = '_6.Movement of Non Performing Assets (NPAs) of Scheduled Commercial Banks (1).xlsx'

# Load Raw Data
table_3_5 = pd.read_excel(file_3_5)
table_restructuring = pd.read_excel(file_restructuring)
table_npa = pd.read_excel(file_npa)

print("Data loaded successfully.")

## Initial Inspection

In [ ]:
def inspect_table(df, name):
    print(f"--- Inspection: {name} ---")
    print(f"Shape: {df.shape}, Columns: {df.columns.tolist()[:5]}...")

inspect_table(table_3_5, "Table 3.5")
inspect_table(table_restructuring, "Restructuring")
inspect_table(table_npa, "NPA Movement")

## Cleaning Utilities

In [ ]:
def to_camel_case(text):
    if pd.isna(text) or text == "": return "unnamedColumn"
    text = str(text)
    words = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', text)
    if not words: words = re.sub(r'[^a-zA-Z0-9]', ' ', text).split()
    if not words: return "unnamedColumn"
    processed = [words[0].lower()]
    for word in words[1:]:
        processed.append(word.capitalize())
    return "".join(processed)

def cleanse_bank_name(val):
    if pd.isna(val): return ""
    s = str(val).upper()
    s = re.sub(r'[^A-Z0-9 ]', '', s)
    return s.strip()

def extract_fiscal_year(val):
    if pd.isna(val): return None
    s = str(val).strip()
    match = re.findall(r'20(\d{2})', s)
    if match: return int("20" + match[-1])
    return None


## First Column Validation & Fix

In [ ]:
def validate_first_column(df):
    if df.empty: return df
    first_col = df.columns[0]
    if df[first_col].isna().all():
        df = df.drop(columns=[first_col])
    elif pd.api.types.is_numeric_dtype(df[first_col]):
        df[first_col] = df[first_col].fillna(df[first_col].mean())
    return df

table_3_5 = validate_first_column(table_3_5)
table_restructuring = validate_first_column(table_restructuring)
table_npa = validate_first_column(table_npa)

## Unnamed Column Renaming

In [ ]:
def infer_column_names(df):
    new_columns = list(df.columns)
    semantic_map = {
        'occupation': 'occupation', 'accounts': 'noOfAccounts',
        'limit': 'creditLimit', 'outstanding': 'amountOutstanding',
        'restructured': 'restructuredAmount', 'loan': 'loanId'
    }
    for i, col in enumerate(new_columns):
        if "Unnamed" in str(col):
            inferred = None
            for val in df.iloc[:15, i]:
                val_str = str(val).lower()
                for key, mapped in semantic_map.items():
                    if key in val_str:
                        inferred = mapped; break
                if inferred: break
                if len(val_str) > 2 and not val_str.replace('.','').isdigit():
                    inferred = val_str.strip(); break
            if inferred: new_columns[i] = inferred
    df.columns = new_columns
    return df

table_3_5 = infer_column_names(table_3_5)
table_restructuring = infer_column_names(table_restructuring)
table_npa = infer_column_names(table_npa)

## Row-Level Cleaning

In [ ]:
def clean_row_levels(df):
    if 1 in df.index:
        row_1_vals = df.loc[1]
        new_cols = list(df.columns)
        for i, val in enumerate(row_1_vals):
            if pd.notna(val) and str(val).strip() != "" and ("Unnamed" in str(new_cols[i]) or "unnamed" in str(new_cols[i]).lower()):
                new_cols[i] = str(val).strip()
        df.columns = new_cols
    if 2 in df.index:
        df = df.drop(index=2)
    return df

table_3_5 = clean_row_levels(table_3_5)
table_restructuring = clean_row_levels(table_restructuring)
table_npa = clean_row_levels(table_npa)

## Column Name Standardization

In [ ]:
def standardize_columns(df):
    df.columns = [to_camel_case(col) for col in df.columns]
    new_cols = []
    counts = {}
    for col in df.columns:
        if col in counts:
            counts[col] += 1
            new_cols.append(f"{col}_{counts[col]}")
        else:
            counts[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

table_3_5 = standardize_columns(table_3_5)
table_restructuring = standardize_columns(table_restructuring)
table_npa = standardize_columns(table_npa)

## Master Consolidation

In [ ]:
# Sub-Process 1.1: Temporal Normalization
def apply_temporal(df, name):
    year_col = next((col for col in df.columns[:3] if any(x in col.lower() for x in ['year', 'march', 'unnamed'])), df.columns[0])
    df['fiscalYear'] = df[year_col].apply(extract_fiscal_year).ffill()
    df = df[df['fiscalYear'] >= 2018].copy()
    df['fiscalYear'] = df['fiscalYear'].astype(int)
    print(f"Unique fiscalYear values for {name}: {sorted(df['fiscalYear'].unique())}")
    return df

table_3_5 = apply_temporal(table_3_5, "Table 3.5")
table_restructuring = apply_temporal(table_restructuring, "Restructuring")
table_npa = apply_temporal(table_npa, "NPA Movement")

# Sub-Process 1.2: Entity Resolution
bank_col_npa = next((col for col in table_npa.columns if any(x in col.lower() for x in ['bank', 'scheduled'])), table_npa.columns[1])
table_npa['bankName'] = table_npa[bank_col_npa].map(cleanse_bank_name)
master_bank_list = [b for b in table_npa['bankName'].unique() if len(b) > 3]

def resolve_banks(df, master_list):
    df = df.copy()
    bank_col = next((col for col in df.columns if 'bank' in col.lower() and col != 'bankName'), df.columns[1])
    df['rawBankName'] = df[bank_col].map(cleanse_bank_name)
    unique_names = [n for n in df['rawBankName'].unique() if n]
    mapping = {name: process.extractOne(name, master_list, processor=utils.default_process)[0] 
               if name and len(name) > 3 and process.extractOne(name, master_list, processor=utils.default_process)[1] > 80 
               else name for name in unique_names if name}
    df['bankName'] = df['rawBankName'].map(mapping)
    return df

table_restructuring = resolve_banks(table_restructuring, master_bank_list)

# Sub-Process 1.3: Sectoral Aggregation
def aggregate_3_5(df):
    val_cols = [col for col in df.columns if any(x in col.lower() for x in ['outstanding', 'limit'])]
    id_cols = ['fiscalYear', 'occupation']
    melted = pd.melt(df, id_vars=id_cols, value_vars=val_cols, var_name='attr', value_name='val')
    melted['bankGroup'] = melted['attr'].apply(lambda x: 'Public' if 'public' in x.lower() else ('Private' if 'private' in x.lower() else 'Foreign'))
    features = melted.pivot_table(index=['fiscalYear', 'bankGroup'], columns='occupation', values='val', aggfunc='sum').reset_index()
    features.columns = [to_camel_case(f"credit_{c}") if c not in ['fiscalYear', 'bankGroup'] else c for c in features.columns]
    return features

df_3_5_features = aggregate_3_5(table_3_5)

# Sub-Process 1.4: Incremental Left-Join
def assign_group(name):
    if any(x in str(name).upper() for x in ['STATE BANK', 'CANARA', 'PUNJAB', 'INDIAN', 'BARODA', 'CENTRAL']): return 'Public'
    return 'Private'

table_npa['bankGroup'] = table_npa['bankName'].apply(assign_group)
npa_num_cols = table_npa.select_dtypes(include=[np.number]).columns
npa_target_col = npa_num_cols[-1] if len(npa_num_cols) > 0 else table_npa.columns[-1]
table_npa['npaClosingBalance'] = pd.to_numeric(table_npa[npa_target_col], errors='coerce').fillna(0)

rest_val_cols = table_restructuring.select_dtypes(include=[np.number]).columns
rest_col = rest_val_cols[-1] if len(rest_val_cols) > 0 else table_restructuring.columns[-1]
table_restructuring['restructuredAmountValue'] = pd.to_numeric(table_restructuring[rest_col], errors='coerce').fillna(0)

master_df = pd.merge(table_npa[['fiscalYear', 'bankName', 'bankGroup', 'npaClosingBalance']], 
                     table_restructuring[['fiscalYear', 'bankName', 'restructuredAmountValue']], 
                     on=['fiscalYear', 'bankName'], how='left')

master_df = pd.merge(master_df, df_3_5_features, on=['fiscalYear', 'bankGroup'], how='left')
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

# Sub-Process 1.5: Missing Value Propagation
sector_cols = [c for c in master_df.columns if c.startswith('credit')]
master_df['isImputed'] = 0
for col in sector_cols:
    mask = master_df[col].isnull()
    master_df.loc[mask, 'isImputed'] = 1
    master_df[col] = master_df[col].fillna(master_df.groupby(['fiscalYear', 'bankGroup'])[col].transform('mean'))

# Final Validation & Save
num_cols_val = master_df.select_dtypes(include=[np.number]).columns
master_df[num_cols_val] = master_df[num_cols_val].astype(np.float32)
master_df.to_csv('Master_Bank_Data_Consolidated.csv', index=False)


## PHASE 2: FINANCIAL FEATURE ENGINEERING

In [ ]:
# Sub-Process 2.1: Mathematical Ratio Calculation
def safe_divide(num, den):
    if den == 0: return 0.0
    return float(num / den)

credit_cols = [col for col in master_df.columns if col.startswith('credit')]
master_df['totalAdvances'] = master_df[credit_cols].sum(axis=1)

# NPA Ratio
master_df['npaRatio'] = master_df.apply(lambda r: safe_divide(r['npaClosingBalance'], r['totalAdvances']), axis=1)

# Risk Weight Ratio
limit_cols = [col for col in master_df.columns if 'limit' in col.lower() and col.startswith('credit')]
out_cols = [col for col in master_df.columns if 'outstanding' in col.lower() and col.startswith('credit')]
master_df['totalLimit'] = master_df[limit_cols].sum(axis=1)
master_df['totalOutstanding'] = master_df[out_cols].sum(axis=1)
master_df['riskWeightRatio'] = master_df.apply(lambda r: safe_divide(r['totalLimit'], r['totalOutstanding']), axis=1)

# Restructuring Stress
master_df['totalAssets'] = master_df['totalAdvances'] * 1.2
master_df['restructuringStress'] = master_df.apply(lambda r: safe_divide(r['restructuredAmountValue'], r['totalAssets']), axis=1)

# Sub-Process 2.2: Target Label Synthesis
master_df['isHighRisk'] = (master_df['npaRatio'] > 0.05).astype(int)

# Sub-Process 2.3: Feature Selection & Cleaning
raw_cols = [col for col in master_df.columns if any(x in col.lower() for x in ['balance', 'limit', 'outstanding', 'amount', 'total', 'credit'])]
cols_to_drop = [c for c in raw_cols if c not in ['fiscalYear', 'bankName', 'bankGroup']]
df_final = master_df.drop(columns=cols_to_drop)

# Impute new NaNs in ratios
ratio_cols = ['npaRatio', 'riskWeightRatio', 'restructuringStress']
for col in ratio_cols:
    df_final[col] = df_final[col].fillna(df_final.groupby('bankGroup')[col].transform('median'))


## PHASE 3: THE PYTORCH DATA PIPELINE

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Sub-Process 3.1: Chronological Data Splitting
df_final = df_final.sort_values(by=['fiscalYear', 'bankName']).reset_index(drop=True)
train_df = df_final[df_final['fiscalYear'] <= 2023].copy()
test_df = df_final[df_final['fiscalYear'] >= 2024].copy()

# Sub-Process 3.2: Scaling & Tensor Transformation
X_cols = [col for col in df_final.columns if col not in ['fiscalYear', 'bankName', 'bankGroup', 'isHighRisk']]
scaler = StandardScaler()
scaler.fit(train_df[X_cols])

X_train_t = torch.tensor(scaler.transform(train_df[X_cols]), dtype=torch.float32)
X_test_t = torch.tensor(scaler.transform(test_df[X_cols]), dtype=torch.float32)

y_train_t = torch.tensor(train_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(test_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)

# Sub-Process 3.3: Custom BankDefaultDataset Class
class BankDefaultDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): 
        return len(self.X)
    def __getitem__(self, idx): 
        return self.X[idx], self.y[idx]

# Sub-Process 3.4: DataLoader Configuration
train_dataset = BankDefaultDataset(X_train_t, y_train_t)
test_dataset = BankDefaultDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)


## PHASE 4: ARTIFICIAL NEURAL NETWORK ARCHITECTURE

In [ ]:
# Sub-Process 4.1: Model Class Definition
class CreditRiskANN(nn.Module):
    def __init__(self, input_dim):
        super(CreditRiskANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=0.3),
            nn.Linear(32, 1)
        )
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            if m.out_features == 1: nn.init.xavier_normal_(m.weight)
            else: nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, x): return self.model(x)

input_dim = X_train_t.shape[1]
model = CreditRiskANN(input_dim).to(device)


## PHASE 5: MODEL TRAINING

In [ ]:
# Sub-Process 5.1: Hyperparameters & Initialization
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Sub-Process 5.2: Training Loop
epochs = 50
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Training Loss: {avg_loss:.4f}")


## PHASE 6: MODEL EVALUATION

In [ ]:
# Sub-Process 6.1: Evaluation Mode
model.eval()
all_preds = []
all_labels = []

# Sub-Process 6.2: Prediction Loop
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        
        # Forward pass (logits)
        logits = model(batch_features)
        
        # Sigmoid -> Classes
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

# Sub-Process 6.3: Metrics Calculation
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print("\n--- Model Evaluation (Test Set) ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
